# NeSy-Mamba v8 Training Analysis

**v8a** = EMA gate with static α (per-slot learnable constant)  
**v8b** = EMA gate with dynamic α: α_k(t) = σ(W_α · h_t + b_αk) — input-dependent memory gate

**Key question:** Do slots now vary across input examples? (The v7 smoking gun was all slots identical at ~0.5518)

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

# ══════════════════════════════════════════════════════════════════
# PASTE v8 RESULTS HERE (from Kaggle logs)
# ══════════════════════════════════════════════════════════════════

epochs = list(range(1, 21))

# ── v7 baseline (for comparison) ─────────────────────────────────
v7_val_acc   = [0.557, 0.556, 0.604, 0.658, 0.675, 0.805, 0.813, 0.811, 0.814, 0.817,
                0.817, 0.815, 0.815, 0.815, 0.812, 0.813, 0.809, 0.812, 0.808, 0.809]
v7_collapsed = [0.571]*20
v7_gate_mean = [0.258, 0.261, 0.258, 0.262, 0.265, 0.263, 0.260, 0.260, 0.261, 0.260,
                0.261, 0.263, 0.261, 0.265, 0.263, 0.266, 0.265, 0.260, 0.260, 0.266]
v7_entropy   = [0.357, 0.356, 0.351, 0.361, 0.360, 0.359, 0.357, 0.350, 0.356, 0.351,
                0.353, 0.354, 0.349, 0.353, 0.352, 0.355, 0.354, 0.346, 0.346, 0.357]
v7_depths    = {0: 0.931, 1: 0.691, 2: 0.763, 3: 0.774, 4: 0.602, 5: 0.500}

# ── v8b results (FILL FROM KAGGLE LOGS) ─────────────────────────
# TODO: paste actual values after kernel completes
v8_train_acc = []  # 20 values
v8_val_acc   = []  # 20 values
v8_task_loss = []  # 20 values
v8_lr_vals   = []  # 20 values

# Slot diagnostics
v8_gate_mean  = []  # 20 values
v8_collapsed  = []  # 20 values
v8_frac_above = []  # 20 values
v8_entropy    = []  # 20 values

# Per-class accuracy
v8_acc_true  = []  # 20 values
v8_acc_false = []  # 20 values

# Per-depth accuracy at best epoch
v8_depths = {}  # {0: X, 1: X, 2: X, 3: X, 4: X, 5: X}

# Self-explanation demo slot values (5 examples)
# v7 had ALL identical: [0.5518, 0.5518, 0.5518] — the smoking gun
v8_demo_slots = []  # list of 5 lists, each with 7 slot values

print('Data loaded. Fill in v8 values from Kaggle logs.')
print(f'v7 best val acc: {max(v7_val_acc):.4f}')

## Figure 1: Accuracy — v7 (monotonic) vs v8 (EMA dynamic α)

# ── Figure 1: Accuracy Comparison ────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1a: Val accuracy comparison
ax = axes[0]
ax.plot(epochs, v7_val_acc, 'orange', marker='s', markersize=4, label='v7 (monotonic)', alpha=0.8)
if v8_val_acc:
    ax.plot(epochs, v8_val_acc, 'steelblue', marker='o', markersize=4, label='v8b (EMA dynamic α)')
ax.axhline(y=0.544, color='gray', ls='--', alpha=0.5, label='Majority class')
ax.set_xlabel('Epoch'); ax.set_ylabel('Val Accuracy')
ax.set_title('Validation Accuracy: v7 vs v8')
ax.legend(loc='lower right'); ax.grid(True, alpha=0.3)
ax.set_ylim(0.5, 0.95)

# 1b: Train accuracy comparison
ax = axes[1]
if v8_train_acc:
    ax.plot(epochs, v8_train_acc, 'steelblue', marker='o', markersize=4, label='v8b train')
if v8_val_acc:
    ax.plot(epochs, v8_val_acc, 'steelblue', marker='s', markersize=4, label='v8b val', ls='--')
ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy')
ax.set_title('v8b Train vs Val')
ax.legend(); ax.grid(True, alpha=0.3)
ax.set_ylim(0.5, 0.95)

# 1c: Task loss
ax = axes[2]
if v8_task_loss:
    ax.plot(epochs, v8_task_loss, 'g-o', markersize=4, label='v8b task loss')
    ax2 = ax.twinx()
    if v8_lr_vals:
        ax2.plot(epochs, [lr*1000 for lr in v8_lr_vals], 'purple', ls='--', alpha=0.7)
        ax2.set_ylabel('LR (×1e-3)', color='purple')
ax.set_xlabel('Epoch'); ax.set_ylabel('Task Loss', color='g')
ax.set_title('v8b Loss & LR Schedule')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('v8_accuracy_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure 1 saved.')

## Figure 2: SLOT HEALTH — The Critical Test

**v7 problem:** gate_mean flat at 0.26, collapsed=57.1% constant, entropy flat at 0.35  
**v8 goal:** gate_mean should CHANGE over epochs, collapsed should DECREASE, entropy should INCREASE

# ── Figure 2: Slot Health — v7 vs v8 ─────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 2a: Gate mean over epochs
ax = axes[0]
ax.plot(epochs, v7_gate_mean, 'orange', marker='s', markersize=4, label='v7 (monotonic)', alpha=0.8)
if v8_gate_mean:
    ax.plot(epochs, v8_gate_mean, 'steelblue', marker='o', markersize=4, label='v8b (EMA dynamic α)')
ax.set_xlabel('Epoch'); ax.set_ylabel('Gate Mean')
ax.set_title('Gate Mean Activation')
ax.legend(); ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1)

# 2b: Collapsed fraction over epochs
ax = axes[1]
ax.plot(epochs, v7_collapsed, 'orange', marker='s', markersize=4, label='v7 (monotonic)', alpha=0.8)
if v8_collapsed:
    ax.plot(epochs, v8_collapsed, 'steelblue', marker='o', markersize=4, label='v8b (EMA dynamic α)')
ax.set_xlabel('Epoch'); ax.set_ylabel('Fraction Collapsed')
ax.set_title('Slot Collapse Rate')
ax.legend(); ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1)
ax.axhline(y=0, color='green', ls=':', alpha=0.5, label='Goal: 0%')

# 2c: Slot entropy over epochs
ax = axes[2]
ax.plot(epochs, v7_entropy, 'orange', marker='s', markersize=4, label='v7 (monotonic)', alpha=0.8)
if v8_entropy:
    ax.plot(epochs, v8_entropy, 'steelblue', marker='o', markersize=4, label='v8b (EMA dynamic α)')
ax.axhline(y=np.log(7), color='gray', ls='--', alpha=0.5, label=f'Max entropy (ln7={np.log(7):.2f})')
ax.set_xlabel('Epoch'); ax.set_ylabel('Slot Entropy')
ax.set_title('Slot Diversity (Entropy)')
ax.legend(); ax.grid(True, alpha=0.3)
ax.set_ylim(0, 2.5)

plt.tight_layout()
plt.savefig('v8_slot_health.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure 2 saved.')

## Figure 3: Per-Depth Accuracy — v7 vs v8

# ── Figure 3: Per-Depth Comparison ────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 6))

depths = [0, 1, 2, 3, 4, 5]
depth_labels = [f'd{d}' for d in depths]
n_examples = [7606, 4651, 2824, 2154, 118, 18]
v7_vals = [v7_depths.get(d, 0) for d in depths]

x = np.arange(len(depths))
w = 0.35

bars1 = ax.bar(x - w/2, v7_vals, w, label='v7 (monotonic, 81.7%)', color='orange', alpha=0.8)
if v8_depths:
    v8_vals = [v8_depths.get(d, 0) for d in depths]
    bars2 = ax.bar(x + w/2, v8_vals, w, label='v8b (EMA dynamic α)', color='steelblue', alpha=0.8)

ax.axhline(y=0.544, color='gray', ls='--', alpha=0.5, label='Majority baseline')
ax.set_xticks(x)
ax.set_xticklabels([f'{dl}\n(n={n})' for dl, n in zip(depth_labels, n_examples)])
ax.set_ylabel('Val Accuracy')
ax.set_title('Per-Depth Accuracy: v7 (Monotonic) vs v8b (EMA Dynamic α)')
ax.legend(loc='upper right'); ax.grid(True, alpha=0.3, axis='y')
ax.set_ylim(0.0, 1.05)

# Value labels on v7 bars
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.1%}', ha='center', va='bottom', fontsize=8, color='darkorange')
if v8_depths:
    for bar in bars2:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{bar.get_height():.1%}', ha='center', va='bottom', fontsize=8, color='steelblue')

plt.tight_layout()
plt.savefig('v8_depth_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure 3 saved.')

## Figure 4: Self-Explanation Demo — The Smoking Gun Test

**v7 result:** ALL 5 examples had identical slot values [0.5518, 0.5518, 0.5518]  
**v8 goal:** Each example should have DIFFERENT slot patterns

# ── Figure 4: Self-Explanation Slot Values ────────────────────────
if v8_demo_slots:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # v7 demo (all identical)
    ax = axes[0]
    v7_demo = [[0.5518]*3 + [0]*4] * 5  # 3 alive at 0.5518, 4 dead
    im1 = ax.imshow(v7_demo, cmap='RdYlGn', vmin=0, vmax=1, aspect='auto')
    ax.set_xlabel('Slot Index')
    ax.set_ylabel('Example')
    ax.set_title('v7 (Monotonic): All Examples Identical!')
    ax.set_xticks(range(7))
    ax.set_xticklabels([f'Slot {i}' for i in range(7)], rotation=45)
    ax.set_yticks(range(5))
    ax.set_yticklabels([f'Ex {i+1}' for i in range(5)])
    for i in range(5):
        for j in range(7):
            ax.text(j, i, f'{v7_demo[i][j]:.2f}', ha='center', va='center',
                    fontsize=9, color='black' if v7_demo[i][j] > 0.3 else 'gray')
    plt.colorbar(im1, ax=ax, label='Slot Activation')
    
    # v8 demo (should be different per example)
    ax = axes[1]
    im2 = ax.imshow(v8_demo_slots, cmap='RdYlGn', vmin=0, vmax=1, aspect='auto')
    ax.set_xlabel('Slot Index')
    ax.set_ylabel('Example')
    ax.set_title('v8b (EMA Dynamic α): Input-Dependent!')
    ax.set_xticks(range(7))
    ax.set_xticklabels([f'Slot {i}' for i in range(7)], rotation=45)
    ax.set_yticks(range(5))
    ax.set_yticklabels([f'Ex {i+1}' for i in range(5)])
    for i in range(len(v8_demo_slots)):
        for j in range(7):
            ax.text(j, i, f'{v8_demo_slots[i][j]:.2f}', ha='center', va='center',
                    fontsize=9, color='black' if v8_demo_slots[i][j] > 0.3 else 'gray')
    plt.colorbar(im2, ax=ax, label='Slot Activation')
    
    plt.suptitle('Self-Explanation: Slot Activations per Example', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('v8_slot_diversity.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    # Compute diversity metrics
    slot_array = np.array(v8_demo_slots)
    print(f'\nSlot diversity metrics:')
    print(f'  Per-slot std across examples: {slot_array.std(axis=0).round(4)}')
    print(f'  Mean std: {slot_array.std(axis=0).mean():.4f}')
    print(f'  v7 comparison: ALL zeros (identical across examples)')
else:
    print('Waiting for v8 self-explanation demo results...')
    print('Paste v8_demo_slots from Kaggle logs.')

## Figure 5: Evolution Summary — v6 → v7 → v8

# ── Figure 5: Multi-Version Comparison ────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

versions = ['v6\n(early)', 'v7\n(monotonic)', 'v8b\n(EMA dynamic α)']
colors = ['#FF9800', '#F44336', '#2196F3']

# Fill in v8 values when available
v8_best_acc = max(v8_val_acc) if v8_val_acc else 0
v8_best_collapsed = min(v8_collapsed) if v8_collapsed else 1

# 5a: Accuracy progression
ax = axes[0]
accs = [0.660, 0.817, v8_best_acc]
bars = ax.bar(versions, accs, color=colors, alpha=0.85, edgecolor='black', linewidth=0.5)
ax.axhline(y=0.544, color='gray', ls='--', alpha=0.5, label='Majority')
ax.set_ylabel('Best Val Accuracy')
ax.set_title('Accuracy Progression')
ax.set_ylim(0.5, 1.0)
ax.grid(True, alpha=0.3, axis='y')
for bar, acc in zip(bars, accs):
    if acc > 0:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{acc:.1%}', ha='center', va='bottom', fontsize=12, fontweight='bold')

# 5b: Slot health progression
ax = axes[1]
collapsed_vals = [0.714, 0.571, v8_best_collapsed]
bars = ax.bar(versions, collapsed_vals, color=colors, alpha=0.85, edgecolor='black', linewidth=0.5)
ax.set_ylabel('Fraction Collapsed (lower = better)')
ax.set_title('Slot Collapse Progression')
ax.set_ylim(0, 1.0)
ax.grid(True, alpha=0.3, axis='y')
ax.axhline(y=0, color='green', ls=':', alpha=0.5)
for bar, val in zip(bars, collapsed_vals):
    if val > 0:
        label = f'{val:.1%}\n({int(val*7)}/7 dead)'
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                label, ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.suptitle('NeSy-Mamba Evolution: Accuracy + Interpretability', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('v8_evolution_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure 5 saved.')

## Summary Table

# ── Final Summary ─────────────────────────────────────────────────
print('='*75)
print('  NeSy-Mamba COMPARISON: v6 → v7 → v8b')
print('='*75)
print(f'{"Metric":<25} {"v6":>12} {"v7 (mono)":>12} {"v8b (EMA-dyn)":>15}')
print('-'*75)
print(f'{"Val Accuracy":<25} {"66.0%":>12} {"81.7%":>12} {(f"{max(v8_val_acc):.1%}" if v8_val_acc else "TBD"):>15}')
print(f'{"Slots Alive":<25} {"2/7":>12} {"3/7 (same)":>12} {"TBD":>15}')
print(f'{"Collapsed":<25} {"71.4%":>12} {"57.1%":>12} {(f"{min(v8_collapsed):.1%}" if v8_collapsed else "TBD"):>15}')
print(f'{"Gate Mean":<25} {"0.095":>12} {"0.260":>12} {(f"{v8_gate_mean[-1]:.3f}" if v8_gate_mean else "TBD"):>15}')
print(f'{"Entropy":<25} {"~0.15":>12} {"0.351":>12} {(f"{v8_entropy[-1]:.3f}" if v8_entropy else "TBD"):>15}')
print(f'{"Input-dependent slots":<25} {"No":>12} {"No":>12} {"TBD":>15}')
print(f'{"Gate Mode":<25} {"monotonic":>12} {"monotonic":>12} {"EMA dyn-α":>15}')
print(f'{"Parameters":<25} {"~270K":>12} {"~273K":>12} {"~273K+":>15}')
print('='*75)

if v8_val_acc:
    delta_acc = max(v8_val_acc) - 0.817
    print(f'\n  Accuracy change vs v7: {"+" if delta_acc >= 0 else ""}{delta_acc:.1%}')
if v8_collapsed:
    print(f'  Collapse change vs v7: {min(v8_collapsed) - 0.571:+.1%}')
    
print('\n  KEY QUESTION: Do v8 slots vary across examples?')
if v8_demo_slots:
    slot_arr = np.array(v8_demo_slots)
    mean_std = slot_arr.std(axis=0).mean()
    if mean_std > 0.05:
        print(f'  ✅ YES! Mean cross-example std = {mean_std:.4f} (v7 was 0.0000)')
    else:
        print(f'  ❌ NO. Mean cross-example std = {mean_std:.4f} — still collapsed')
else:
    print('  ⏳ Waiting for results...')